In [ ]:
import numpy as np
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install sentence-transformers
!pip install faiss-gpu-cu12

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/zalo-dataset/corpus.csv')

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
import os 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device for embedding: {device}")
embedding_model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder', device=device)

if device == 'cuda':
     embedding_model.half()

valid_chunks = (
    "Title: " + df['title'].astype(str) + 
    ". Content: " + df['text'].astype(str)
).tolist()

with torch.no_grad():
     doc_embeddings = embedding_model.encode(
         valid_chunks,
         batch_size=512,
         convert_to_numpy=True
     )

doc_embeddings = doc_embeddings.astype('float32')

dimension = doc_embeddings.shape[1]

doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

if device == 'cuda' and len(valid_chunks) > 0:
     print("Creating FAISS index on GPU.")
     res = faiss.StandardGpuResources()
     cpu_index = faiss.IndexFlatIP(dimension)
     gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
     gpu_index.add(doc_embeddings)
     final_index = faiss.index_gpu_to_cpu(gpu_index)

else:
     print("Creating FAISS index on CPU.")
     final_index = faiss.IndexFlatIP(dimension)
     final_index.add(doc_embeddings)

index_filename = "index.faiss"

output_dir = "/kaggle/working/" 
save_path = os.path.join(output_dir, index_filename)

print(f"Saving FAISS index to: {save_path}")
faiss.write_index(final_index, save_path)
print("FAISS index saved successfully!")